# Reprezentacje wektorowe tekstu

Na dzisiejszych zajęciach zapoznamy się z metodami reprezentacji znaczenia tekstów, rozumianych jako sekwencje tokenów. Metody oparte o uczenie maszynowe zakładają reprezentacje o charakterze wektorowym, tj. takie, gdzie istotne cechy badanych obiektów są modelowane jako wymiary w przestrzeni wielowymiarowej, zaś obiektom przypisane są punkty w tejże przestrzeni.

Z naszego punktu widzenia, istotne będzie rozróżnienie na wektory rzadkie (zawierające wiele zer, i najczęściej liczby naturalne na pozostałych pozycjach) oraz wektory gęste (praktycznie bez zer, najczęściej zawierające liczby rzeczywiste).

Najważniejsze biblioteki z których dzisiaj skorzystamy to:
1. `sklearn` - biblioteka zawierająca funkcjonalności do reprezentacji, analizy, i manipulacji danymi, wraz z klasycznymi narzędziami uczenia maszynowego
1. `numpy` - biblioteka służąca do reprezentacji macierzy, wektorów, oraz przetwarzania ich zgodnie z zasadami algebry liniowej

In [ ]:
import pandas
import numpy
from pathlib import Path

DATA_DIR = Path("../../data")

Wczytamy dane z shared task'a dot. rozpoznawania emocji wyrażonych w komentarzach i recenzjach konsumenckich (https://github.com/poleval/2024-emotion-recognition).

In [ ]:
EMO_DIR = DATA_DIR / "2024-emotion-recognition" / "train"
emo_file = EMO_DIR / "in.tsv"
emo_df = pandas.read_csv(emo_file, sep="\t")
emo_df

W jaki sposób możemy przypisać wektory tekstom?

Jest wiele odpowiedzi na to pytanie. Probierzem tego, jak dobra jest dana metoda będzie to, czy geometria przestrzeni w której będą reprezentowane teksty będzie odzwierciedlać relacje które chcemy modelować. Jednym ze sposobów operacjonalizacji tego kryterium będzie żądanie, aby teksty *podobne* do siebie w sensie lingwistycznym (np. pod względem znaczenia) były *bliskie* w przestrzeni wektorowej (i *vice versa*).

Najprostszą metodą wektoryzacji jest podejście *Bag of Words* (BOW). Polega ono na "wrzuceniu" słów występujących w tekście do "worka", i.e. zignorowaniu struktury tekstu, i skupieniu jedynie na liczbie wystąpień każdego ze słów w danym tekście. Aby tego dokonać musimy wpierw:
1. zbudować słownik (i.e. zbiór wszystkich słów które będziemy chcieli reprezentować)
2. przypisać każdemu ze słów identyfikator liczbowy, korespondujący z wymiarem w przestrzeni wektorowej
3. przypisać każdemu z tekstów wektor w taki sposób, że na *n*-tej pozycji będzie się znajdowała liczba wystąpień słowa o identyfikatorze *n* w tym tekście

Zadania te implementuje w ramach `sklearn` klasa `CountVectorizer` za pomocą metody `fit_transform`:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
vectors = cv.fit_transform(emo_df["text"])
vectors

Na wyjściu otrzymujemy macierz rzadką reprezentującą 7169 tekstów w przestrzeni o 24699 wymiarach. Macierze rzadkie są przeważnie reprezentowane w pamięci komputera inaczej niż macierze gęste. Pamiętanie każdego zera z osobna byłoby ogromnym marnotrawstwem, więc korzystamy z formatów skompresowanych, w tym wypadku `CSR`. Pełna wersja tej macierzy (w precyzji `float32`) zajęłaby $ \frac{7169 \times 2699 \times 4B}{1000^{2}} \approx 708 MB $. 

**Pracując z macierzami rzadkimi należy pilnować, by nie doprowadzić do konwersji ich do postaci pełnej, co może skutkować przepełnieniem pamięci na maszynie!**

Zbadajmy niektóre własności tego obiektu:

In [ ]:
num_objs, num_features = vectors.shape
frac_zero = (vectors == 0).mean()
print(f"{(frac_zero * 100):.5}% of values == 0")

Obiekt `CountVectorizer` przechowuje także mapowanie między tokenami i identyfikatorami (numerami wymiarów):

In [ ]:
vocab = cv.vocabulary_
inv_vocab = cv.get_feature_names_out()
vocab_sample = {inv_vocab[i]: i for i in range(1000, 1050)}
print(vocab_sample)

Wektory możemy analizować ręcznie, np. aby znaleźć najczęściej występujące słowo, należy dla każdej z cech (kolumn) sprawdzić sumę wartości, i znaleźć indeks najwyższej wartości, a następnie słowo, które z nim koresponduje:

In [ ]:
word_freq = vectors.sum(axis=0)
most_freq_id = word_freq.argmax()
print(inv_vocab[most_freq_id], word_freq[0, most_freq_id])

## Ćwiczenie 1:
Znajdź słowo, które najczęściej występuje w tych samych dokumentach co słowo "bardzo"

In [ ]:
# TODO

Jak wspomnieliśmy wyżej, naszym podstawowym żądaniem jest zdolność do modelowania podobieństwa pod względem lingwistycznym. W szczególności, dla większości zadań interesowałoby nas podobieństwo znaczenia. Sprawdźmy więc, korzystając z podobieństwa kosinusowego jako metryki, jak wygląda pod tym względem nasza przestrzeń wektorowa na przykładzie:

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

reference_index = 0
sims = cosine_similarity(vectors, vectors)
top_inds = sims[reference_index].argsort()[-5:][::-1] # indices of 5 most similar texts to the reference one
print(sims[reference_index, top_inds])
emo_df.iloc[top_inds]

Widzimy, że podobieństwo które udało nam się tu uchwycić jest bardzo powierzchowne. Jako najbliższego sąsiada dla:

`2 gwiazdki.` 

otrzymaliśmy:

`4 gwiazdki?` 

teksty podobne **leksykalnie**, ale o diametralnie różnym znaczeniu. BOW okazuje się bardzo toporną metodą. Problem ten ma też "rewers" - teksty semantycznie ekwiwalentne, mogą być zdaniem modelu BOW kompletnie niepowiązane.`CountVectorizer` może przetworzyć nowe teksty - bez douczania (`transform` to `fit_transform` bez uczenia), wykorzystajmy to:

In [ ]:
equiv_text = "dwa punkty."
equiv_vec = cv.transform([equiv_text])
cosine_similarity(vectors[0], equiv_vec)

Pierwszą kardynalną wadą metody BOW jest więc całkowita niezdolność do modelowania relacji semantycznych między różnymi leksykalnie jednostkami.

Drugą wadą kardynalną, jest niemożliwość reprezentacji szyku zdania (i kolejności elementów w ogóle, a więc także innych struktur tekstualnych):

In [ ]:
bad_text = "brak zalet dużo wad"
good_text = "brak wad dużo zalet"
bad = cv.transform([bad_text])
good = cv.transform([good_text])
print((good == bad).toarray().all())
print(cosine_similarity(bad, good))

Tę drugą wadę możemy próbować zażegnać poprzez proste udoskonalenie. Metoda BOW nie musi operować na pojedynczych tokenach, jako cechy tekstu możemy patrzeć na tzw. *n-gramy* - uporządkowane n-tki tokenów:

In [ ]:
pair_cv = CountVectorizer(ngram_range=(2,2)) # represent only pairs of tokens
pair_vectors = pair_cv.fit_transform([good_text, bad_text])
pair_vocab = pair_cv.vocabulary_
print(pair_vocab)
print(cosine_similarity(pair_vectors[0], pair_vectors[1]))

Niestety stosowanie n-gramów szybko prowadzi do eksplozji kombinatorycznej (dla par, słownik będzie w pesymistycznym przypadku rósł kwadratowo wraz z liczbą unikalnych słów), i słabo sprawuje się w językach o swobodnym szyku (jak język polski) gdzie powiązane relacją gramatyczną tokeny mogą (w teorii) być od siebie oddalone o dowolną liczbę tokenów pośrednich.

Zobaczmy jakie są rozmiary bigramowej reprezentacji:

In [ ]:
pair_vectors = pair_cv.fit_transform(emo_df["text"])
pair_vocab = pair_cv.vocabulary_
print(f"2-gram vocabulary is {len(pair_vocab) / len(vocab):.5} times bigger than unigram vocabulary.")
zero_cells = 0
all_cells = 0
for vec in pair_vectors:
    zero_cells += (vec == 0).sum()
    all_cells += vec.shape[1]

pair_frac_zero = zero_cells / all_cells
print(f"{(pair_frac_zero * 100):.5}% of values == 0")

## Ćwiczenie 2:

Zaproponuj tekst, które będzie wyrażał bardzo podobną treść do tego, oznaczonego jako `good_text`, i używał wszystkich pojawiających się w nim tokenów (plus potencjalnie inne tokeny), a jednak w reprezentacji 2-gramowej będzie oceniony jako kompletenie doń niepodobny (i.e. podobieństwo kosinusowe będzie równe 0).

In [ ]:
# TODO

Trzecią kardynalną wadą, jest równe traktowanie każdego ze słów. Język, jak się nieraz przekonamy (i przekonaliśmy np. obserwując prawo Zipfa), jest dziedziną w której na wielu płaszczyznach będziemy się spotykali z radykalną heterogenią. W tym wypadku, jedne słowa są zasadniczo bardziej dystynktywne, podczas gdy inne - przez swoje występowanie nie mówią nam w zasadzie nic o samym tekście w którym się pojawiają. Spójrzmy na przykład patologiczny:

In [ ]:
degenerate_text = "i jednak nawet nie tak"
degenerate_vec = cv.transform([degenerate_text])
degenerate_sim = cosine_similarity(degenerate_vec, vectors)
degenerate_top_inds = degenerate_sim[0].argsort()[-5:][::-1] # indices of 5 most similar texts to the reference one
degenerate_top = emo_df.iloc[degenerate_top_inds]
degenerate_top["similarity"] = [degenerate_sim[0][i] for i in degenerate_top_inds]
degenerate_top

Ponieważ podobieństwo kosinusowe jest symetryczne, nasz patologiczny przykład mógłby wkraść się jako jeden z najbardziej podobnych tekstów do otoczenia jednego z całkowicie normalnych tekstów zanieczyszczając je. 

Patologia naszego przykładu polega na tym, iż składa się on w całości z tzw. *wyrazów funkcyjnych* - *function words* (które przeciwstawiane są *wyrazom semantycznym* - *content words*). Wyrazy funkcyjne to m.in. przyimki, spójniki, partykuły, zaimki. Są one neutralne dziedzinowo i semantycznie, i odgrywają rolę "rusztowania" na którym buduje się struktura gramatyczna łącząca wyrazy semantyczne. Jeżeli zależy nam tylko na semantyce, to uzasadnionym jest pominięcie ich. 

Częstym zabiegiem w klasycznym NLP było definiowanie krótkiej listy słów które chcemy pomijać jako nieistotne - **stoplisty**, często lądowały na niej właśnie wyrazy funkcyjne. Popularna biblioteka `spaCy` dla języka polskiego definiuje następującą stoplistę:

In [ ]:
STOP_WORDS = set("""
a aby ach acz aczkolwiek aj albo ale alez
ależ ani az aż

bardziej bardzo beda bede bedzie bez bo bowiem by
byc byl byla byli bylo byly bym bynajmniej być był
była było były będzie będą będę

cala cali caly cała cały chce choć ci cie
ciebie cię co cokolwiek coraz cos coś czasami czasem czemu
czy czyli często

daleko dla dlaczego dlatego do dobrze dokad dokąd
dosc dość duzo dużo dwa dwaj dwie dwoje dzis
dzisiaj dziś

gdy gdyby gdyz gdyż gdzie gdziekolwiek gdzies gdzieś go
godz

i ich ile im inna inne inny
innych iv ix iz iż

ja jak jakas jakaś jakby jaki jakichs jakichś jakie
jakis jakiz jakiś jakiż jakkolwiek jako jakos jakoś je jeden
jedna jednak jednakze jednakże jedno jednym jedynie jego jej jemu
jesli jest jestem jeszcze jezeli jeśli jeżeli juz już ją

kazdy każdy kiedy kierunku kilka kilku kims kimś kto
ktokolwiek ktora ktore ktorego ktorej ktory ktorych ktorym ktorzy ktos
ktoś która które którego której który których którym którzy ku

lecz lub

ma mają mam mamy mało mi miał miedzy
mimo między mna mnie mną moga mogą moi moim moj
moja moje moze mozliwe mozna może możliwe można mu musi
my mój

na nad nam nami nas nasi nasz nasza nasze
naszego naszych natomiast natychmiast nawet nia nic nich nie niech
niego niej niemu nigdy nim nimi niz nią niż no

o obok od ok około on ona one
oni ono oraz oto owszem

pan pana pani po pod podczas pomimo ponad
poniewaz ponieważ powinien powinna powinni powinno poza prawie przeciez
przecież przed przede przedtem przez przy

raz razie roku rowniez również

sam sama sie się skad skąd soba sobie sobą
sposob sposób swoje są

ta tak taka taki takich takie takze także tam
te tego tej tel temu ten teraz też to toba
tobie tobą totez toteż totobą trzeba tu tutaj twoi twoim
twoj twoja twoje twym twój ty tych tylko tym tys
tzw tę

u

vi vii viii

w wam wami was wasi wasz wasza wasze we
według wie wiele wielu więc więcej wlasnie wszyscy wszystkich wszystkie
wszystkim wszystko wtedy wy właśnie wśród

xi xii xiii xiv xv

z za zaden zadna zadne zadnych zapewne zawsze zaś
ze zeby znow znowu znów zostal został

żaden żadna żadne żadnych że żeby""".split()
)

Wszystkie słowa składające się na wskazany przykład patologiczny wpadają do stoplisty, więc zostanie im przypisany wektor zerowy, a zatem zerowe podobieństwo do innych tekstów:

In [ ]:
scv = CountVectorizer(stop_words=list(STOP_WORDS))
svectors = scv.fit_transform(emo_df["text"]) 
sdegenerate_vec = scv.transform([degenerate_text])
sdegenerate_sim = cosine_similarity(sdegenerate_vec, svectors)
sdegenerate_top_inds = sdegenerate_sim[0].argsort()[-5:][::-1] # indices of 5 most similar texts to the reference one
print(sdegenerate_sim[0, sdegenerate_top_inds])
emo_df.iloc[sdegenerate_top_inds]

Stoplisty są jednak rozwiązaniem mało subtelnym - stawiają ostry podział tam, gdzie w rzeczywistości granica nie jest wcale ostra. Ponadto są one przeważnie definiowane niezależnie od dziedziny, tymczasem w niektórych zastosowaniach będziemy chcieli zwrócić uwagę na cechy które są dystynktywne W RAMACH tej dziedziny. 

Bardziej eleganckie, i elastyczne jest postawienie tego problemu w sposób, który penalizuje słowa funkcyjne w sposób ciągły, oraz motywowany danymi. Rozwiązaniem które próbuje zrealizować to podejście jest reprezentacja **TF-IDF**. Przypisuje on każdej parze n-ty tekst - k-te słowo (i.e. każdej komórce w macierzy) wartość zadaną wzorem:

$\frac{TF}{DF} = f(w_{k}, t_{n}) \times (log(\frac{N+1}{N_{k}+1}) + 1)$

gdzie $ f(w_{k}, t_{n})$ oznacza liczbę wystąpień k-tego słowa w n-tym tekście, $N_{k}$ oznacza liczbę dokumentów w których występuje k-te słowo, zaś $N$ oznacza całkowitą liczbę dokumentów w korpusie. 

TF-IDF jest w sklearn zaimplementowany analogicznie jak `CountVectorizer`:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
tvectors = tf.fit_transform(emo_df["text"])

Wyraz $DF$ penalizuje rolę słów (cech) które są mało dystynktywne dla tekstów. Słowa funkcyjne będą obecne powszechnie, ponieważ gramatyka pozostaje niezmieniona niezależnie od tematu tekstów:

In [ ]:
function_word = "oraz" 
function_ind = tf.vocabulary_[function_word]
content_word = "zaskoczony"
content_ind = tf.vocabulary_[content_word]
tf.idf_[[function_ind, content_ind]]

Należy zwrócić uwagę na to, że $DF$ jest liczony na podstawie globalnych własności korpusu. Oznacza to, że dodanie nowych tekstów może zmienić sposób w jaki reprezentowane są stare teksty. Najczęściej będzie to się wiązało z poprawą reprezentacji: powiększenie korpusu wygładzi wpływ losowości, i pozwoli lepiej oszacować częstość słów.

## Ćwiczenie 3: 

Korzystając z atrybutu `.idf_` utwórz (jako tabelę `DataFrame`) ranking słów pod kątem ich "wszędobylskości", oraz sprawdź jak ma się to do obecności na zdefiniowanej powyżej stopliście


In [ ]:
# TODO

## Reprezentacje Gęste

Powyższe metody są metodami leksykalnymi - każda z cech koresponduje z jednostką ze słownika. Zaletą takiego podejścia jest jasna interpretowalność cech. Wadą jest niezdolność do reprezentacji relacji pomiędzy tymi cechami, i redundancji która z nich wynika.

W istocie w języku mamy do czynienia z całym wachlarzem różnego rodzaju relacji między słowami (synonimia, derywacja, fleksja) które z punktu widzenia BoW lub TfIDF są kompletnie niewidoczne. Reprezentacje gęste kompresują wiedzę o tożsamości słów do postaci ustalonej długości wektorów poprzez trening na dużych korpusach tekstów. Cechy tracą jasną interpretowalność, ale jednocześnie lepiej uchwytują subtelności słownika.

Wczytajmy pobrane z zewnątrz, gotowe, embeddingi słów - macierz rozmiaru $V \times d$ gdzie $d$ to liczba używanych cech, zaś $V$ to wielkość słownika.

Korzystamy z embeddingów wytrenowanych przez IPI PAN: 

https://dsmodels.nlp.ipipan.waw.pl/

http://dsmodels.nlp.ipipan.waw.pl/dsmodels/wiki-forms-all-100-skipg-ns-30-it100.txt.gz

In [ ]:
w2v_path = DATA_DIR / "others" / "wiki-forms-all-100-skipg-ns-30-it100.txt"

w2v = pandas.read_csv(w2v_path, sep=" ", skiprows=[0], header=None)
emb_arr = numpy.vstack(w2v.set_index(0).apply(numpy.array, axis=1))
ind2w = w2v[0].to_dict()
w2ind = {w: ind for ind, w in ind2w.items()}

Zdefiniujmy kilka pomocniczych funkcji do pracy z naszą macierzą. Zwróćmy uwagę, na powracający problem OOV: słowa które nie znajdują się w słowniku naszych embeddingów (a więc nie było ich w korpusie treningowym) w ogóle nie mają reprezentacji, i będziemy je musieli pomijać. Oznacza to, że jeśli pracujemy z danymi o specyficznym słownictwie, gotowe embeddingi mogą słabo odzwierciedlać semantykę takiej dziedziny.

In [ ]:
DIM = 100

def get_vec(word):
    ind = w2ind.get(word, None)
    if ind:
        return emb_arr[ind]
    return numpy.zeros((DIM))

def most_similar(word):
    vec = get_vec(word)
    sims = cosine_similarity([vec], emb_arr)[0]
    best_inds = sims.argsort()[-11:-1][::-1]
    best_words = [ind2w[i] for i in best_inds]
    best_scores = sims[best_inds]
    ranking = list(zip(best_words, best_scores))
    return ranking

word = "pies"
print(word, get_vec(word))
most_similar(word)

Spróbujmy zinterpretować pierwsze dwa wymiary naszej macierzy - jakie słowa korespondują z najwyższymi, a jakie z najniższymi wartościami tych cech? Dla wygody znormalizujmy wektory:

In [ ]:
from matplotlib import pyplot as plt

vec_lengths = numpy.linalg.norm(emb_arr, axis=1).reshape(-1, 1)
norm_emb_arr = emb_arr / vec_lengths

two_feats = norm_emb_arr[:,0:2]
tops = two_feats.argsort(axis=0)
low, high = tops[:5], tops[-5:]
inds = numpy.concat([low, high]).flatten()
labs = [ind2w[i] for i in inds]
x, y = two_feats[inds].T
plt.scatter(x,y)
for lab, px, py in zip(labs, x, y):
    plt.annotate(lab, (px, py), fontsize=8)

plt.show()

Trudno wskazać sensowną interpretację.

Teraz spróbujmy dodatkowo skompresować informację korzystając z PCA (zrobimy to na surowych wektorach):

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(2)
reduced = pca.fit_transform(emb_arr)
tops = reduced.argsort(axis=0)
low, high = tops[:5], tops[-5:]
inds = numpy.concat([low, high]).flatten()
labs = [ind2w[i] for i in inds]
x, y = reduced[inds].T
plt.scatter(x,y)
for lab, px, py in zip(labs, x, y):
    plt.annotate(lab, (px, py), fontsize=8)

plt.show()

## Ćwiczenie 4

Potraktuj podobieństwa semantyczne do słów "jasny" i "zły" jako cechy. Oblicz poziom tych cech dla każdego ze słów w słowniku. Następnie wybierz skrajne obserwacje dla każdej z tych cech, i oznacz je na wykresie. Jak zintepretujesz te wyniki? Jak to działa dla innej, wybranej przez Ciebie pary słów?

In [ ]:
# TODO

Word2Vec jest metodą reprezentacji *słów*. TF-IDF pozwala na uzyskanie reprezentacji *tekstów*. Jak możemy ekstrapolować word2vec tak, by uzyskać stałej długości wektor reprezentujący semantykę tekstu?

Jest wiele metod agregacji sekwencji wektorów do jednego wektora, który próbuje w jakiś sposób oddawać cechy tej sekwencji. Najprostszą z nich jest uśrednienie wektorów. Spróbujmy zobaczyć jak to działa:

In [ ]:
import re
import os

tokenizer = re.compile(r"\b\w+\b")

def text_to_vec(text):
    tokens = tokenizer.findall(text)
    if not tokens:
        return numpy.zeros(DIM)
    vecs = [get_vec(tok) for tok in tokens]
    arr = numpy.vstack(vecs)
    mean_vec = arr.mean(axis=0)
    return mean_vec


dense_vectors = numpy.vstack(emo_df["text"].apply(text_to_vec))
topk = 10
text2text_sims = cosine_similarity(dense_vectors, dense_vectors)
example_ind = 1000
print(emo_df.iloc[example_ind])
top_inds = text2text_sims[example_ind].argsort(descending=True)[1:topk]
pandas.DataFrame({"text": emo_df.iloc[top_inds]["text"], "sim": sims[example_ind, top_inds]})

Jak widzimy udało się nam uzyskać narzędzie pozwalające na poprawne szacowanie podobieństwa znaczenia tekstów bez odwołania do pokrycia leksykalnego.

## Ćwiczenie 5

Oblicz długość wektora każdego ze słów w słowniku załadowanych embeddingów (i.e. utwórz wektor długości $V$ zawierający długości normy l2 każdego z wektorów słów). Znajdź słowa korespondujące z najkrótszymi, i najdłuższymi wektorami. Czy jesteś w stanie zinterpretować tę różnicę? Jak może ona wpływać na reprezentacje tekstów (w odróżnieniu od słów)? Jak ma się ona do stosowanych przez nas metryk odległości?

In [ ]:
# TODO

Jak wspominaliśmy, problem OOV może sprawiać, że trudno będzie uzyskać dobre reprezentacje dla tekstów o charakterystyce odbiegającej od tych, na jakich trenowane były embeddingi. O ile posiadamy dostatecznie dużo własnych danych, możemy jednak wytrenować własne embeddingi. Wczytajmy korpus tekstów specyficznych, ale jednocześnie homogenicznych pod względem tematyki i języka: dziewiętnastowieczne powieści Honoriusza Balzaca:

In [ ]:
import re


DATA_DIR = Path("../../data")
books_dir = DATA_DIR / "lektury" / "balzac"
filepaths = []
for dir_path, subdirs, files in os.walk(books_dir):
    for file in files:
        if file.endswith(".txt"):
            filepath = Path(dir_path) / file
            filepaths.append(filepath)

def read_file(fpath): # helper function
    with open(fpath) as f:
        txt = f.read()
    return txt

balzac_df = pandas.DataFrame({"fpath": filepaths})
balzac_df["text"] = balzac_df["fpath"].apply(read_file)

Aby wytrenować najprostsze embeddingi **word2vec** musimy wpierw podzielić korpus na "zdania". Nie są to koniecznie zdania w sensie gramatycznym - chodzi raczej o krótkie teksty, które zachowują względną ciągłośc znaczenia i tematu. Chodzi tu o to, by pojęcie sąsiedztwa można było uznać za symptom zachodzenia związków semantycznych których raczej nie będziemy się spodziewać pomiędzy np. osobnymi rozdziałami.

In [ ]:
tokenizer = re.compile(r"\b\w+\b")
lines = balzac_df["text"].apply(lambda x: x.split("\n")).explode()
lines = lines[lines.apply(lambda line: len(line.strip()) > 0)]
sentences = lines.apply(lambda x: tokenizer.findall(x.lower())).tolist()
print(f"{len(sentences)} sentences which consist of {sum([len(sent) for sent in sentences])} tokens.")

Algorytm word2vec posiada szybką implementację w bibliotece `gensim`. Aby wytrenować model, skorzystamy z klasy `Word2Vec` do której podamy kilka hiperparametrów:
1. `vector_size` - liczbę wymiarów przestrzeni
1. `epochs` - liczbę epok, i.e. ile razy model zobaczy cały nasz korpus
1. `window` - szerokośc okna, i.e. kontekstu na który będzie mógł patrzeć

Wytrenujmy model i sprawdźmy jak dobrze modeluje podobieństwo semantyczne:

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(sentences=sentences, vector_size=15, epochs=40, window=10, workers=16)
wv = model.wv
wv.most_similar_cosmul("oko")

## Ćwiczenie 6

Hiperparametry które podaliśmy do modelu, nie zostały wybrane w sposób przemyślany. Zastanów się, jak możesz ocenić jakość wytrenowanych embeddingów a następnie:
1. przygotuj kilka przykładów testowych, które pozwolą Ci ocenić ilościowo na ile dobry jest model
2. zaimplementuj grid-search na hiperparametrach (i.e. zaproponuj możliwe wartości hiperparametrów, i wypróbuj wszystkie ich kobinacje)
3. oceń każdy z wytrenowanych modeli, zbierz wyniki eksperymentu w tabeli `DataFrame` i wybierz najlepszy

In [ ]:
# TODO